# Computer Vision Analysis of Vincent van Gogh Paintings

This notebook implements a simple, presentation-friendly pipeline for analysing a collection of Vincent van Gogh paintings.

Main goals:
- extract dominant colour palettes,
- compute colour characteristics such as saturation, brightness and contrast,
- find paintings that are visually similar in terms of colour,
- cluster paintings based on colour features,
- visualise the results for the final presentation and ERK-style paper.

The pipeline is intentionally based on standard Python libraries: `numpy`, `opencv`, `matplotlib`, `pandas`, and `scikit-learn`.

## 1. Setup and automatic dataset download

This notebook is designed so that it can work as a single-file project. By default, it can download a collection of Vincent van Gogh paintings from WikiArt into the local `images/` folder and then run the analysis pipeline.

Recommended folder structure after running the download cell:

```text
project/
  VanGogh_Color_Analysis_Notebook.ipynb
  images/
    1888_sunflowers_204565.jpg
    ...
  output_vangogh/
```

Optional metadata is saved automatically into `metadata.csv`.


In [ ]:
from pathlib import Path
import math
import os
import re
import time
import json
import urllib.parse
from urllib.request import urlopen, Request
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

# Show plots inside the notebook
%matplotlib inline

# === USER SETTINGS ===
IMAGE_DIR = Path("images")              # folder with downloaded Van Gogh paintings
METADATA_PATH = Path("metadata.csv")      # created automatically by the download cell
OUTPUT_DIR = Path("output_vangogh")
PALETTE_K = 5                            # number of dominant colours per image
N_CLUSTERS = None                        # None = choose automatically; or set e.g. 4
TOP_SIMILAR_PAIRS = 20
DOWNLOAD_DATASET = True                    # set False if images are already downloaded
MAX_IMAGES_TO_DOWNLOAD = 350               # use None for all available WikiArt images

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Download the Van Gogh dataset from WikiArt

This cell downloads painting metadata through the WikiArt read-only API and saves images into `IMAGE_DIR`. The analysis below then works only with local files, so you do not need a separate dataset folder beforehand.

Notes:
- The dataset is used for educational / research purposes.
- `MAX_IMAGES_TO_DOWNLOAD` is intentionally limited for faster development. Increase it later if needed.
- If the download fails because of a network issue, run the cell again or manually place images into `images/`.


In [ ]:
def slugify(text: str, max_len: int = 70) -> str:
    text = str(text or "unknown").lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text[:max_len] or "unknown"


def http_json(url: str, timeout: int = 30):
    req = Request(url, headers={"User-Agent": "Mozilla/5.0 (educational image analysis project)"})
    with urlopen(req, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def download_url(url: str, target_path: Path, timeout: int = 45) -> bool:
    req = Request(url, headers={"User-Agent": "Mozilla/5.0 (educational image analysis project)"})
    try:
        with urlopen(req, timeout=timeout) as response:
            data = response.read()
        if len(data) < 1000:
            return False
        target_path.write_bytes(data)
        return True
    except Exception as exc:
        print(f"[warning] Could not download {url}: {exc}")
        return False


def get_first_existing(row: dict, keys: List[str], default=None):
    for key in keys:
        value = row.get(key)
        if value not in [None, "", []]:
            return value
    return default


def fetch_wikiart_paintings(artist_url: str = "vincent-van-gogh") -> List[dict]:
    api_url = (
        "https://www.wikiart.org/en/App/Painting/PaintingsByArtist"
        f"?artistUrl={urllib.parse.quote(artist_url)}&json=2"
    )
    paintings = http_json(api_url)
    if not isinstance(paintings, list):
        raise RuntimeError("Unexpected WikiArt API response. Expected a list of paintings.")
    return paintings


def download_wikiart_artist_dataset(
    image_dir: Path,
    metadata_path: Path,
    artist_url: str = "vincent-van-gogh",
    max_images: Optional[int] = 350,
    sleep_seconds: float = 0.15,
) -> pd.DataFrame:
    image_dir.mkdir(parents=True, exist_ok=True)
    paintings = fetch_wikiart_paintings(artist_url)
    if max_images is not None:
        paintings = paintings[:max_images]

    rows = []
    print(f"Found {len(paintings)} metadata records to process.")

    for i, p in enumerate(paintings, start=1):
        image_url = get_first_existing(p, ["image", "imageUrl", "paintingUrl"])
        title = get_first_existing(p, ["title", "Title"], f"painting_{i}")
        year = get_first_existing(p, ["completitionYear", "completionYear", "year", "yearAsString"])
        content_id = get_first_existing(p, ["contentId", "id", "paintingId"], i)
        style = get_first_existing(p, ["style", "Style"])
        genre = get_first_existing(p, ["genre", "Genre"])
        painting_url = get_first_existing(p, ["url", "paintingUrl"], "")

        if not image_url:
            continue

        ext = Path(urllib.parse.urlparse(image_url).path).suffix.lower()
        if ext not in [".jpg", ".jpeg", ".png", ".webp"]:
            ext = ".jpg"

        filename = f"{slugify(year)}_{slugify(title)}_{content_id}{ext}"
        target = image_dir / filename

        if not target.exists():
            ok = download_url(image_url, target)
            time.sleep(sleep_seconds)
        else:
            ok = True

        if ok:
            rows.append({
                "filename": filename,
                "title": title,
                "year": year,
                "style": style,
                "genre": genre,
                "source_url": image_url,
                "wikiart_page": painting_url,
            })

        if i % 50 == 0:
            print(f"Processed {i}/{len(paintings)} records, downloaded/available: {len(rows)}")

    df_meta = pd.DataFrame(rows)
    df_meta.to_csv(metadata_path, index=False)
    print(f"Saved {len(df_meta)} image records to {metadata_path}")
    print(f"Images are stored in: {image_dir.resolve()}")
    return df_meta


if DOWNLOAD_DATASET:
    metadata_downloaded = download_wikiart_artist_dataset(
        image_dir=IMAGE_DIR,
        metadata_path=METADATA_PATH,
        artist_url="vincent-van-gogh",
        max_images=MAX_IMAGES_TO_DOWNLOAD,
    )
else:
    print("DOWNLOAD_DATASET is False. Using existing local images.")


## 3. Helper functions

These functions load the dataset, extract colour features, create dominant palettes and prepare numerical feature vectors for clustering and similarity search.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

@dataclass
class ImageRecord:
    filename: str
    path: Path
    title: str
    year: Optional[float] = None
    period: Optional[str] = None


def list_images(image_dir: Path) -> List[Path]:
    """Return supported image files from a directory recursively."""
    files = [p for p in image_dir.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS]
    return sorted(files)


def load_metadata(metadata_path: Optional[Path]) -> pd.DataFrame:
    """Load optional metadata CSV. Returns an empty DataFrame if not provided."""
    if metadata_path is None:
        return pd.DataFrame()
    if not metadata_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {metadata_path}")
    df = pd.read_csv(metadata_path)
    if "filename" not in df.columns:
        raise ValueError("Metadata CSV must contain a 'filename' column.")
    return df


def build_records(image_dir: Path, metadata_path: Optional[Path]) -> List[ImageRecord]:
    """Connect image files with optional metadata."""
    images = list_images(image_dir)
    meta = load_metadata(metadata_path)
    meta_by_name: Dict[str, dict] = {}
    if not meta.empty:
        meta_by_name = {str(row["filename"]): row.to_dict() for _, row in meta.iterrows()}

    records: List[ImageRecord] = []
    for path in images:
        row = meta_by_name.get(path.name, {})
        title = str(row.get("title", path.stem.replace("_", " ").replace("-", " ")))
        year = row.get("year", None)
        try:
            year = float(year) if pd.notna(year) else None
        except Exception:
            year = None
        period = row.get("period", None)
        period = str(period) if period is not None and pd.notna(period) else None
        records.append(ImageRecord(filename=path.name, path=path, title=title, year=year, period=period))
    return records


def read_rgb(path: Path, max_side: int = 900) -> Optional[np.ndarray]:
    """Read image as RGB and resize large images for faster processing."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return img


def sample_pixels(img_rgb: np.ndarray, max_pixels: int = 20000) -> np.ndarray:
    """Sample pixels for efficient colour statistics."""
    pixels = img_rgb.reshape(-1, 3)
    if len(pixels) <= max_pixels:
        return pixels.astype(np.float32)
    rng = np.random.default_rng(42)
    idx = rng.choice(len(pixels), size=max_pixels, replace=False)
    return pixels[idx].astype(np.float32)


def extract_dominant_palette(img_rgb: np.ndarray, k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    """Extract dominant colours with K-Means. Returns RGB colours and their proportions."""
    pixels = sample_pixels(img_rgb, max_pixels=25000)
    k = min(k, len(pixels))
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(pixels)
    centers = kmeans.cluster_centers_.clip(0, 255)
    counts = np.bincount(labels, minlength=k).astype(np.float64)
    proportions = counts / counts.sum()
    order = np.argsort(-proportions)
    return centers[order], proportions[order]


def color_histogram_lab(img_rgb: np.ndarray, bins: Tuple[int, int, int] = (8, 8, 8)) -> np.ndarray:
    """Compute normalized Lab colour histogram as a robust colour-similarity feature."""
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    hist = cv2.calcHist([lab], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    hist = hist.flatten().astype(np.float32)
    hist /= (hist.sum() + 1e-9)
    return hist


def color_statistics(img_rgb: np.ndarray) -> Dict[str, float]:
    """Compute basic colour characteristics in HSV and grayscale space."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h, s, v = cv2.split(hsv)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    rgb = img_rgb.astype(np.float32)

    return {
        "mean_r": float(rgb[:, :, 0].mean()),
        "mean_g": float(rgb[:, :, 1].mean()),
        "mean_b": float(rgb[:, :, 2].mean()),
        "mean_hue": float(h.mean()),
        "std_hue": float(h.std()),
        "mean_saturation": float(s.mean()),
        "std_saturation": float(s.std()),
        "mean_brightness": float(v.mean()),
        "std_brightness": float(v.std()),
        "contrast_gray": float(gray.std()),
        "aspect_ratio": float(img_rgb.shape[1] / img_rgb.shape[0]),
        "width": int(img_rgb.shape[1]),
        "height": int(img_rgb.shape[0]),
    }


def palette_feature(palette: np.ndarray, proportions: np.ndarray, k: int) -> np.ndarray:
    """Flatten palette and proportions into one feature vector."""
    pal = palette.astype(np.float32) / 255.0
    if len(pal) < k:
        pad = np.zeros((k - len(pal), 3), dtype=np.float32)
        pal = np.vstack([pal, pad])
        proportions = np.concatenate([proportions, np.zeros(k - len(proportions))])
    return np.concatenate([pal.flatten(), proportions.astype(np.float32)])

## 4. Load and preview the dataset

This step scans the image folder and displays a small sample. If this cell shows zero images, check the `IMAGE_DIR` path.

In [ ]:
records = build_records(IMAGE_DIR, METADATA_PATH)
records_by_filename = {r.filename: r for r in records}

print(f"Found {len(records)} images in {IMAGE_DIR.resolve()}")

# Preview a few images
sample_records = records[:12]
if sample_records:
    cols = 4
    rows = math.ceil(len(sample_records) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, rec in zip(axes, sample_records):
        img = read_rgb(rec.path, max_side=500)
        if img is not None:
            ax.imshow(img)
        ax.set_title(rec.title[:35], fontsize=8)
    plt.tight_layout()
else:
    print("No images found. Put image files into the images/ folder or change IMAGE_DIR.")

## 5. Extract colour features

For each painting, the notebook extracts:
- dominant colour palette,
- average RGB values,
- hue, saturation and brightness statistics,
- grayscale contrast,
- Lab colour histogram for similarity search.

In [ ]:
def make_feature_table(records: List[ImageRecord], out_dir: Path, k_palette: int):
    rows = []
    feature_vectors = []
    hist_vectors = []

    for i, rec in enumerate(records, start=1):
        img = read_rgb(rec.path)
        if img is None:
            print(f"[warning] Could not read: {rec.path}")
            continue

        palette, proportions = extract_dominant_palette(img, k=k_palette)
        stats = color_statistics(img)
        hist = color_histogram_lab(img)

        row = {
            "filename": rec.filename,
            "title": rec.title,
            "year": rec.year,
            "period": rec.period,
            **stats,
        }
        for j in range(k_palette):
            if j < len(palette):
                r, g, b = palette[j]
                prop = proportions[j]
            else:
                r, g, b, prop = 0, 0, 0, 0
            row[f"palette_{j+1}_r"] = float(r)
            row[f"palette_{j+1}_g"] = float(g)
            row[f"palette_{j+1}_b"] = float(b)
            row[f"palette_{j+1}_proportion"] = float(prop)
        rows.append(row)

        stats_vec = np.array([
            stats["mean_r"], stats["mean_g"], stats["mean_b"],
            stats["mean_hue"], stats["std_hue"],
            stats["mean_saturation"], stats["std_saturation"],
            stats["mean_brightness"], stats["std_brightness"],
            stats["contrast_gray"],
        ], dtype=np.float32)
        pal_vec = palette_feature(palette, proportions, k_palette)
        feature_vectors.append(np.concatenate([stats_vec / 255.0, pal_vec, hist]))
        hist_vectors.append(hist)

        if i % 50 == 0:
            print(f"Processed {i}/{len(records)} images")

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No readable images were found. Check IMAGE_DIR.")

    feature_matrix = np.vstack(feature_vectors).astype(np.float32)
    hist_matrix = np.vstack(hist_vectors).astype(np.float32)
    df.to_csv(out_dir / "color_features.csv", index=False)
    np.save(out_dir / "feature_matrix.npy", feature_matrix)
    np.save(out_dir / "lab_histograms.npy", hist_matrix)
    return df, feature_matrix, hist_matrix


df, feature_matrix, hist_matrix = make_feature_table(records, OUTPUT_DIR, PALETTE_K)
df.head()

## 6. Visualise dominant colour palettes

This gives a first visual overview of the dominant colours in the painting collection.

In [ ]:
def palette_from_row(row: pd.Series, k_palette: int):
    colors = []
    props = []
    for j in range(k_palette):
        colors.append((
            row[f"palette_{j+1}_r"] / 255.0,
            row[f"palette_{j+1}_g"] / 255.0,
            row[f"palette_{j+1}_b"] / 255.0,
        ))
        props.append(row[f"palette_{j+1}_proportion"])
    return colors, props


def show_palette_grid(df: pd.DataFrame, k_palette: int, max_items: int = 20):
    sample = df.head(min(max_items, len(df)))
    fig, axes = plt.subplots(len(sample), 1, figsize=(10, max(3, 0.45 * len(sample))))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        colors, props = palette_from_row(row, k_palette)
        left = 0.0
        for color, prop in zip(colors, props):
            ax.barh([0], [prop], left=left, color=color, edgecolor="white")
            left += prop
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        ax.set_xticks([])
        ax.set_ylabel(str(row["title"])[:35], rotation=0, ha="right", va="center", fontsize=8)
    fig.suptitle("Dominant colour palettes - sample", fontsize=14)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "dominant_palette_grid.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()


def show_global_palette(df: pd.DataFrame, k_palette: int):
    fig, ax = plt.subplots(figsize=(8, 2.2))
    for j in range(k_palette):
        r = df[f"palette_{j+1}_r"].mean() / 255.0
        g = df[f"palette_{j+1}_g"].mean() / 255.0
        b = df[f"palette_{j+1}_b"].mean() / 255.0
        ax.add_patch(plt.Rectangle((j, 0), 1, 1, color=(r, g, b), ec="white"))
    ax.set_xlim(0, k_palette)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Average dominant palette across the collection")
    fig.savefig(OUTPUT_DIR / "global_average_palette.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()

show_global_palette(df, PALETTE_K)
show_palette_grid(df, PALETTE_K, max_items=20)

## 7. Cluster paintings by colour features

Clustering tries to automatically find groups of paintings with similar colour characteristics. If `N_CLUSTERS` is set to `None`, the notebook estimates a reasonable number of clusters using the silhouette score.

In [ ]:
def choose_cluster_count(features_scaled: np.ndarray, min_k: int = 2, max_k: int = 8) -> int:
    n = len(features_scaled)
    if n < 6:
        return max(2, min(n, 3))
    max_k = min(max_k, n - 1)
    best_k = 2
    best_score = -1.0
    for k in range(min_k, max_k + 1):
        labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(features_scaled)
        score = silhouette_score(features_scaled, labels)
        if score > best_score:
            best_k = k
            best_score = score
    return best_k


def cluster_images(df: pd.DataFrame, feature_matrix: np.ndarray, n_clusters: Optional[int]):
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(feature_matrix)
    if n_clusters is None:
        n_clusters = choose_cluster_count(features_scaled)
    labels = KMeans(n_clusters=n_clusters, random_state=42, n_init=30).fit_predict(features_scaled)
    result = df.copy()
    result["cluster"] = labels
    return result, n_clusters


df_clustered, selected_k = cluster_images(df, feature_matrix, N_CLUSTERS)
print(f"Selected number of clusters: {selected_k}")
df_clustered.to_csv(OUTPUT_DIR / "color_features_with_clusters.csv", index=False)
df_clustered[["filename", "title", "cluster", "mean_saturation", "mean_brightness", "contrast_gray"]].head()

## 8. Cluster summary

This table helps interpret the clusters. For example, one cluster may contain brighter paintings, another darker or more saturated ones.

In [ ]:
cluster_summary = df_clustered.groupby("cluster").agg(
    count=("filename", "count"),
    mean_saturation=("mean_saturation", "mean"),
    mean_brightness=("mean_brightness", "mean"),
    mean_contrast=("contrast_gray", "mean"),
    mean_hue=("mean_hue", "mean"),
).reset_index()

cluster_summary.to_csv(OUTPUT_DIR / "cluster_summary.csv", index=False)
cluster_summary

## 9. Saturation and brightness scatter plot

This is a simple visualisation of colour characteristics. Each point is one painting, and the colour of the point represents the assigned cluster.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    df_clustered["mean_saturation"],
    df_clustered["mean_brightness"],
    c=df_clustered["cluster"],
    s=40,
    alpha=0.75,
)
ax.set_xlabel("Mean saturation")
ax.set_ylabel("Mean brightness")
ax.set_title("Colour characteristics of the painting collection")
fig.colorbar(scatter, ax=ax, label="Cluster")
fig.savefig(OUTPUT_DIR / "saturation_brightness_scatter.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

## 10. Find the most colour-similar paintings

Similarity is computed using cosine similarity over the extracted colour feature vectors. The result is a ranked list of painting pairs that are closest in colour-based representation.

In [ ]:
def compute_similarity(df: pd.DataFrame, feature_matrix: np.ndarray, top_n: int = 20):
    sim = cosine_similarity(feature_matrix)
    np.fill_diagonal(sim, -1.0)

    pairs = []
    seen = set()
    flat_order = np.argsort(sim, axis=None)[::-1]
    n = sim.shape[0]
    for idx in flat_order:
        i, j = divmod(idx, n)
        if i == j:
            continue
        key = tuple(sorted((i, j)))
        if key in seen:
            continue
        seen.add(key)
        pairs.append({
            "image_a": df.iloc[i]["filename"],
            "title_a": df.iloc[i]["title"],
            "image_b": df.iloc[j]["filename"],
            "title_b": df.iloc[j]["title"],
            "similarity": float(sim[i, j]),
        })
        if len(pairs) >= top_n:
            break
    return pd.DataFrame(pairs), sim

pairs_df, similarity_matrix = compute_similarity(df_clustered, feature_matrix, TOP_SIMILAR_PAIRS)
pairs_df.to_csv(OUTPUT_DIR / "most_similar_color_pairs.csv", index=False)
np.save(OUTPUT_DIR / "similarity_matrix.npy", similarity_matrix)
pairs_df.head(10)

## 11. Visualise most similar pairs

This contact sheet is useful for the presentation because it makes the similarity search easy to understand.

In [ ]:
def show_similar_pairs(pairs_df: pd.DataFrame, records_by_filename: Dict[str, ImageRecord], max_pairs: int = 6):
    rows = min(max_pairs, len(pairs_df))
    if rows == 0:
        print("No pairs to show.")
        return
    fig, axes = plt.subplots(rows, 2, figsize=(7, rows * 3.1))
    if rows == 1:
        axes = np.array([axes])
    for r in range(rows):
        pair = pairs_df.iloc[r]
        for c, img_col, title_col in [(0, "image_a", "title_a"), (1, "image_b", "title_b")]:
            ax = axes[r, c]
            ax.axis("off")
            filename = pair[img_col]
            rec = records_by_filename.get(filename)
            if rec:
                img = read_rgb(rec.path, max_side=650)
                if img is not None:
                    ax.imshow(img)
            ax.set_title(f"{pair[title_col]}
similarity={pair['similarity']:.3f}", fontsize=8)
    fig.suptitle("Most colour-similar painting pairs", fontsize=14)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "most_similar_pairs_contact_sheet.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()

show_similar_pairs(pairs_df, records_by_filename, max_pairs=6)

## 12. 2D similarity maps using dimensionality reduction

The feature vectors have many dimensions, so they cannot be plotted directly. PCA and t-SNE reduce them to two dimensions. Similar paintings should appear closer together in the resulting map.

In [ ]:
def reduce_dimensions(feature_matrix: np.ndarray):
    scaler = StandardScaler()
    x = scaler.fit_transform(feature_matrix)
    coords = pd.DataFrame(index=np.arange(len(x)))

    pca = PCA(n_components=2, random_state=42)
    pca_xy = pca.fit_transform(x)
    coords["pca_x"] = pca_xy[:, 0]
    coords["pca_y"] = pca_xy[:, 1]

    if len(x) > 5:
        perplexity = min(30, max(5, (len(x) - 1) // 3))
        tsne = TSNE(n_components=2, random_state=42, init="pca", learning_rate="auto", perplexity=perplexity)
        tsne_xy = tsne.fit_transform(x)
        coords["tsne_x"] = tsne_xy[:, 0]
        coords["tsne_y"] = tsne_xy[:, 1]

    # Optional UMAP if installed: pip install umap-learn
    try:
        import umap  # type: ignore
        reducer = umap.UMAP(n_components=2, random_state=42)
        umap_xy = reducer.fit_transform(x)
        coords["umap_x"] = umap_xy[:, 0]
        coords["umap_y"] = umap_xy[:, 1]
    except Exception:
        pass

    return coords


def plot_2d_map(df: pd.DataFrame, coords: pd.DataFrame, method: str):
    x_col = f"{method}_x"
    y_col = f"{method}_y"
    if x_col not in coords.columns or y_col not in coords.columns:
        print(f"{method.upper()} coordinates are not available.")
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(coords[x_col], coords[y_col], c=df["cluster"], s=38, alpha=0.8)
    ax.set_title(f"2D visualisation of colour-based similarities ({method.upper()})")
    ax.set_xlabel(f"{method.upper()} 1")
    ax.set_ylabel(f"{method.upper()} 2")
    fig.colorbar(sc, ax=ax, label="Cluster")
    fig.savefig(OUTPUT_DIR / f"{method}_similarity_map.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()

coords = reduce_dimensions(feature_matrix)
results_with_coords = pd.concat([df_clustered.reset_index(drop=True), coords.reset_index(drop=True)], axis=1)
results_with_coords.to_csv(OUTPUT_DIR / "analysis_results_with_2d_coordinates.csv", index=False)

plot_2d_map(df_clustered, coords, "pca")
plot_2d_map(df_clustered, coords, "tsne")
plot_2d_map(df_clustered, coords, "umap")

## 13. Visualise cluster examples

These image grids help interpret the clusters. In the final paper or video, select the most meaningful clusters and describe what they have in common.

In [ ]:
def show_cluster_contact_sheet(df: pd.DataFrame, records_by_filename: Dict[str, ImageRecord], cluster_id: int, max_items: int = 12):
    group = df[df["cluster"] == cluster_id].head(max_items)
    if group.empty:
        print(f"Cluster {cluster_id} is empty.")
        return
    cols = 4
    rows = math.ceil(len(group) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, (_, row) in zip(axes, group.iterrows()):
        rec = records_by_filename.get(row["filename"])
        if rec:
            img = read_rgb(rec.path, max_side=650)
            if img is not None:
                ax.imshow(img)
        ax.set_title(str(row["title"])[:32], fontsize=8)
    fig.suptitle(f"Cluster {cluster_id}: example paintings", fontsize=14)
    fig.tight_layout()
    cluster_dir = OUTPUT_DIR / "clusters"
    cluster_dir.mkdir(exist_ok=True)
    fig.savefig(cluster_dir / f"cluster_{cluster_id}.png", dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()

for cluster_id in sorted(df_clustered["cluster"].unique()):
    show_cluster_contact_sheet(df_clustered, records_by_filename, cluster_id, max_items=12)

## 14. Simple interpretation helper

Use this section to write short observations after looking at the visual outputs. These notes can later be reused in the presentation video and ERK-style paper.

In [ ]:
print("Possible interpretation notes:")
print("- Which cluster contains brighter paintings?")
print("- Which cluster contains more saturated paintings?")
print("- Do the most similar pairs look visually convincing?")
print("- Are there clusters that correspond to similar motifs, such as landscapes, portraits, or night scenes?")
print("- Are the 2D maps readable and useful for explaining the dataset structure?")

cluster_summary.sort_values("mean_brightness", ascending=False)

## 15. Output files

The notebook saves the main results into the output folder:

```text
output_vangogh/
  color_features.csv
  color_features_with_clusters.csv
  most_similar_color_pairs.csv
  cluster_summary.csv
  analysis_results_with_2d_coordinates.csv
  global_average_palette.png
  dominant_palette_grid.png
  saturation_brightness_scatter.png
  pca_similarity_map.png
  tsne_similarity_map.png
  most_similar_pairs_contact_sheet.png
  clusters/cluster_*.png
```

These files are suitable for the presentation, final paper and video demonstration.